In [1]:
import pandas as pd

## 0. Load Common Metadata And Subset Raw

In [2]:
TYPE_CALIBRATOR = "Calibrator"

samples = pd.read_table(
    "/mnt/data/test/sample.txt"
)
samples_calibrators = samples.loc[
    samples["SampleType"] == TYPE_CALIBRATOR
]
samples_calibrators.to_csv(
    "/mnt/code/preprocess-somascan-data/tests/data/samples.csv",
    index = False
)

features = pd.read_table(
    "/mnt/data/test/somamer.txt"
).rename({"SeqId": "ProbeId"}, axis = 1)
features.to_csv(
    "/mnt/code/preprocess-somascan-data/tests/data/features.csv",
    index = False
)

In [3]:
# Read in raw data
measurements_raw = pd.read_table(
    "/mnt/data/test/RFU_raw.txt",
    header=None
)
measurements_raw.index = samples.set_index(["PlateId", "PlatePosition"]).index
measurements_raw.columns = features["ProbeId"]

# Melt raw data
measurements_raw = (
    measurements_raw.reset_index()
                         .melt(
                             id_vars=["PlateId", "PlatePosition"],
                             var_name="ProbeId"
                         )
)

# Subset expected data
measurements_raw = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_raw.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)

# Write expected data
measurements_raw.to_csv(
    "/mnt/code/preprocess-somascan-data/tests/data/measurements.csv",
    index = False
)

## 1. Hybridization control normalization

In [4]:
# Read in expected data
measurements_expected = pd.read_table(
    "/mnt/data/test/RFU_hyb.txt",
    header=None
)
measurements_expected.index = samples.set_index(["PlateId", "PlatePosition"]).index
measurements_expected.columns = features["ProbeId"]

# Melt expected data
measurements_expected = (
    measurements_expected.reset_index()
                         .melt(
                             id_vars=["PlateId", "PlatePosition"],
                             var_name="ProbeId"
                         )
)

# Subset expected data
measurements_expected = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_expected.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)

# Write expected data
measurements_expected.to_csv(
    "/mnt/code/preprocess-somascan-data/tests/data/measurements_hcn.csv",
    index = False
)

In [5]:
measurements_expected

,PlateId,PlatePosition,ProbeId,value
11,P0031168,A9,10000-28,872.3505
11,P0031168,A9,10001-7,408.4856
11,P0031168,A9,10003-15,241.1457
11,P0031168,A9,10006-25,844.6107
11,P0031168,A9,10008-43,748.6732
...,...,...,...,...
2043,P0031201,H3,9993-11,1953.9630
2043,P0031201,H3,9994-217,2327.9250
2043,P0031201,H3,9995-6,2626.3470
2043,P0031201,H3,9997-12,9028.4260


In [6]:
# Read in test data
measurements_test = pd.read_csv(
    "/mnt/data/processed/measurements.hybridization_control_normalized.csv"
)

# Subset test data
measurements_test = (
    samples_calibrators.loc[:, ["PlateId", "PlatePosition"]]
                       .join(
                           measurements_test.set_index(["PlateId", "PlatePosition"]),
                           on = ["PlateId", "PlatePosition"]
                       )
)

In [7]:
measurements_test

,PlateId,PlatePosition,ProbeId,value
11,P0031168,A9,10000-28,872.350490
11,P0031168,A9,10001-7,408.485553
11,P0031168,A9,10003-15,241.145676
11,P0031168,A9,10006-25,844.610726
11,P0031168,A9,10008-43,748.673202
...,...,...,...,...
2043,P0031201,H3,9993-11,1953.962597
2043,P0031201,H3,9994-217,2327.924744
2043,P0031201,H3,9995-6,2626.347368
2043,P0031201,H3,9997-12,9028.425763


In [8]:
measurements_joined = measurements_expected.set_index(
    ["PlateId", "PlatePosition", "ProbeId"]
).value.rename("value_expected").to_frame().join(
    measurements_test.set_index(
        ["PlateId", "PlatePosition", "ProbeId"]
    ).value.rename("value_test")
)

In [9]:
measurements_joined["value_diff_ppm"] = 1e6 * (
    measurements_joined.value_expected - measurements_joined.value_test
) / measurements_joined.value_expected

measurements_joined.value_diff_ppm.max() < 1

np.True_

## Check test data folder

In [10]:
ls -lh /mnt/code/preprocess-somascan-data/tests/data/

total 47M
-rw-r--r--. 1 ubuntu ubuntu 757K Aug 31 19:39 features.csv
-rw-r--r--. 1 ubuntu ubuntu    0 Aug 31 19:12 __init__.py
-rw-r--r--. 1 ubuntu ubuntu  22M Aug 31 19:39 measurements.csv
-rw-r--r--. 1 ubuntu ubuntu  24M Aug 31 19:39 measurements_hcn.csv
-rw-r--r--. 1 ubuntu ubuntu  26K Aug 31 19:39 samples.csv
